In [1]:
#Importing libraries
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from datasets import Dataset, ClassLabel, Features, Sequence, Value

In [2]:
tok_lab_df=pd.read_csv('E:\\PROJECTS\\Privacy-Risk-Analyser\\data\\token_labels_fixed.csv')
tok_lab_df.head()

,tokens,labels
0,"['▁My', '▁name', '▁is', '▁A', 'al', 'iyah', '▁...","['O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NAME', ..."
1,"['▁My', '▁name', '▁is', '▁Konstantin', '▁Beck'...","['O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NAME', ..."
2,"['▁As', '▁Mi', 'eko', '▁Mitsubishi', ',', '▁an...","['O', 'B-NAME', 'I-NAME', 'I-NAME', 'O', 'O', ..."
3,"['▁My', '▁name', '▁is', '▁Kaz', 'u', 'o', '▁Su...","['O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NAME', ..."
4,"['▁My', '▁name', '▁is', '▁Ar', 'ina', '▁Sun', ...","['O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NAME', ..."


In [3]:
tok_lab=tok_lab_df[:1000]

In [4]:
# Convert list of strings into actual lists
tok_lab["tokens"] = tok_lab["tokens"].apply(ast.literal_eval)
tok_lab["labels"] = tok_lab["labels"].apply(ast.literal_eval)

C:\Users\Dell\AppData\Local\Temp\ipykernel_21632\2760448289.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tok_lab["tokens"] = tok_lab["tokens"].apply(ast.literal_eval)
C:\Users\Dell\AppData\Local\Temp\ipykernel_21632\2760448289.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tok_lab["labels"] = tok_lab["labels"].apply(ast.literal_eval)


In [5]:
# Finding unique labels using set
all_labels=[]
for labels in tok_lab["labels"]:
    for label in labels:
        all_labels.append(label)
all_labels = list(set(all_labels))

# Sorting
all_labels = sorted(all_labels)

# Defining features
features = Features({
    "tokens": Sequence(Value("string")),
    "labels": Sequence(ClassLabel(names=all_labels))
})



In [6]:
# Create the Hugging Face Dataset from the Pandas DataFrame
dataset = Dataset.from_pandas(tok_lab, features=features)

In [7]:
print(dataset)

Dataset({
    features: ['tokens', 'labels'],
    num_rows: 1000
})


In [8]:
print(dataset[0])

{'tokens': ['▁My', '▁name', '▁is', '▁A', 'al', 'iyah', '▁Pop', 'ova', ',', '▁and', '▁I', '▁am', '▁a', '▁je', 'wel', 'er', '▁with', '▁13', '▁years', '▁of', '▁experience', '.', '▁I', '▁remember', '▁a', '▁very', '▁unique', '▁and', '▁challenging', '▁project', '▁I', '▁had', '▁to', '▁work', '▁on', '▁last', '▁year', '.', '▁A', '▁customer', '▁approach', 'ed', '▁me', '▁with', '▁a', '▁precio', 'us', '▁family', '▁', 'heir', 'loom', '▁-', '▁a', '▁dia', 'mond', '▁nec', 'kla', 'ce', '▁that', '▁had', '▁been', '▁passed', '▁down', '▁through', '▁generation', 's', '.', '▁Unfortunately', ',', '▁the', '▁nec', 'kla', 'ce', '▁was', '▁in', '▁poor', '▁condition', ',', '▁with', '▁several', '▁loo', 'se', '▁dia', 'mond', 's', '▁and', '▁a', '▁broken', '▁cla', 'sp', '.', '▁The', '▁customer', '▁wanted', '▁me', '▁to', '▁resto', 're', '▁it', '▁to', '▁its', '▁former', '▁glo', 'ry', ',', '▁but', '▁it', '▁was', '▁clear', '▁that', '▁this', '▁would', '▁be', '▁no', '▁ordina', 'ry', '▁repair', '.', '▁U', 'sing', '▁my', '▁spe

In [10]:
# Finding unique labels using set
all_labels=[]
for labels in tok_lab["labels"]:
    for label in labels:
        all_labels.append(label)
all_labels = list(set(all_labels))

# Sorting
all_labels = sorted(all_labels)

# Defining features
features = Features({
    "tokens": Sequence(Value("string")),
    "labels": Sequence(ClassLabel(names=all_labels))
})



In [9]:
labels = dataset.features["labels"].feature.names

id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}

In [11]:
id2label

{0: 'B-ADDRESS',
 1: 'B-EMAIL',
 2: 'B-NAME',
 3: 'B-PHONE',
 4: 'I-ADDRESS',
 5: 'I-EMAIL',
 6: 'I-NAME',
 7: 'I-PHONE',
 8: 'O'}

In [12]:
label2id

{'B-ADDRESS': 0,
 'B-EMAIL': 1,
 'B-NAME': 2,
 'B-PHONE': 3,
 'I-ADDRESS': 4,
 'I-EMAIL': 5,
 'I-NAME': 6,
 'I-PHONE': 7,
 'O': 8}

In [13]:
from datasets import DatasetDict

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# 80% for training
train_testval = dataset.train_test_split(test_size=0.2, seed=42)

#remaining 20% equally for validation and test (10% each)
test_val = train_testval['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train': train_testval['train'],
    'validation': test_val['train'],
    'test': test_val['test']
})

print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 100
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 100
    })
})


In [14]:
dataset_dict["train"][0]


{'tokens': ['▁Angel',
  '▁Schneider',
  ',',
  '▁a',
  '▁season',
  'ed',
  '▁trans',
  'la',
  'tor',
  '▁with',
  '▁a',
  '▁k',
  'na',
  'ck',
  '▁for',
  '▁',
  'brid',
  'ging',
  '▁language',
  '▁barrier',
  's',
  ',',
  '▁recently',
  '▁completed',
  '▁a',
  '▁re',
  'mark',
  'able',
  '▁job',
  '-',
  'related',
  '▁project',
  '▁that',
  '▁show',
  'ca',
  'sed',
  '▁his',
  '/',
  'her',
  '▁linguis',
  'tic',
  '▁expertise',
  '▁and',
  '▁dedicati',
  'on',
  '▁to',
  '▁accu',
  'ra',
  'cy',
  '.',
  '▁In',
  '▁this',
  '▁project',
  ',',
  '▁Angel',
  '▁was',
  '▁task',
  'ed',
  '▁with',
  '▁trans',
  'la',
  'ting',
  '▁a',
  '▁series',
  '▁of',
  '▁in',
  'trica',
  'te',
  '▁legal',
  '▁documents',
  '▁from',
  '▁German',
  '▁to',
  '▁English',
  '▁for',
  '▁a',
  '▁multi',
  'national',
  '▁corporation',
  '▁based',
  '▁in',
  '▁South',
  '▁Africa',
  '.',
  '▁With',
  '▁a',
  '▁ke',
  'en',
  '▁eye',
  '▁for',
  '▁detail',
  '▁and',
  '▁a',
  '▁deep',
  '▁understan

In [27]:
dataset_dict.save_to_disk("privacy_ner_dataset")




Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]